# 07 — Subject-stratified K-fold evaluation

Wraps **`dysxai_cv_evaluation.py`** (source below). Reports mean ± std **test** AUC across folds for scenarios such as `time_zero`, `time_rep`, `no_time`.

**Requires:** run **`00_initialization.ipynb`** first (or ensure `dysxai_init` paths are valid).


## `dysxai_cv_evaluation.py` — full file (reference)


In [3]:
import os
import sys
from pathlib import Path

from IPython.display import Code, display

root = os.path.abspath(os.getcwd())
if root not in sys.path:
    sys.path.insert(0, root)

display(Code(filename=str(Path(root) / "dysxai_cv_evaluation.py"), language="python"))


"""
Subject-stratified K-fold evaluation with validation + held-out test per fold.

Runs leakage-related scenarios (Time-only, no-Time, padding modes, optional batch padding)
and reports mean/std of **test** AUC across folds. Scaler is fit on training subjects only.

Example:
    python dysxai_cv_evaluation.py --k 5 --epochs 20 --quick
    python dysxai_cv_evaluation.py --k 5 --scenarios time_zero,time_rep,no_time --epochs 50
"""

from __future__ import annotations

import argparse
import csv
import os
import sys
from typing import Dict, List, Tuple

import numpy as np
import torch
from sklearn.model_selection import train_test_split

_PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

import dysxai_init
from dysxai_init import (
    TIME_CHANNEL_INDEX,
    Config,
    subject_stratified_kfold_indices,
)
from dysxai_leakage_ablation import (
    CNN1DModel,
    build_loaders_from_meta,
    indices_without_time,
    train_with_val_and_optional_test,
)

Scenario = Tuple[str, List[int], int, str, str]  # name, feat_idx, in_ch, pad_mode, padding


def resolve_scenarios(spec: str, in_channels: int) -> List[Scenario]:
    """Build scenario list from comma-separated names; ``in_channels`` from data (e.g. 13)."""
    nt = indices_without_time(in_channels)
    built_in: Dict[str, Scenario] = {
        "time_zero": ("time_zero", [TIME_CHANNEL_INDEX], 1, "zero", "fixed"),
        "time_rep": ("time_rep", [TIME_CHANNEL_INDEX], 1, "replicate", "fixed"),
        "time_batch_zero": ("time_batch_zero", [TIME_CHANNEL_INDEX], 1, "zero", "batch"),
        "time_batch_rep": ("time_batch_rep", [TIME_CHANNEL_INDEX], 1, "replicate", "batch"),
        "no_time": ("no_time", nt, len(nt), "zero", "fixed"),
    }
    names = [x.strip() for x in spec.split(",") if x.strip()]
    out: List[Scenario] = []
    for n in names:
        if n not in built_in:
            raise ValueError(f"Unknown scenario '{n}'. Choose from: {list(built_in)}")
        out.append(built_in[n])
    return out


def run_one_fold(
    train_meta,
    val_meta,
    test_meta,
    scenario: Scenario,
    epochs: int,
    device: str,
    verbose: bool,
):
    name, feat_idx, in_ch, pad_mode, padding = scenario
    tl, vl, te, cw, n_ch = build_loaders_from_meta(
        train_meta,
        val_meta,
        test_meta,
        pad_mode=pad_mode,
        padding=padding,
    )
    assert n_ch >= max(feat_idx) + 1
    model = CNN1DModel(in_channels=in_ch)
    _, val_m, test_m = train_with_val_and_optional_test(
        tl,
        vl,
        model,
        epochs,
        cw,
        feat_idx,
        device,
        test_loader=te,
        verbose=verbose,
    )
    return val_m, test_m


def run_kfold(
    n_folds: int = 5,
    num_epochs: int | None = None,
    quick: bool = False,
    seed: int = 42,
    val_fraction: float = 0.2,
    scenarios: str = "time_zero,time_rep,no_time",
    output_csv: str | None = None,
    verbose: bool = True,
) -> None:
    if num_epochs is not None:
        epochs = num_epochs
    elif quick:
        epochs = 15
    else:
        epochs = Config.NUM_EPOCHS
    torch.manual_seed(seed)
    np.random.seed(seed)

    meta_df = dysxai_init.run_init(verbose=False)
    folds, subject_ids, y_sub = subject_stratified_kfold_indices(meta_df, n_folds=n_folds, random_state=seed)

    # Channel count from first fold (train/val split for fold 0)
    tv_idx, te_idx = folds[0]
    tv_subjects = subject_ids[tv_idx]
    tv_y = y_sub[tv_idx]
    tr_rel, va_rel = train_test_split(
        np.arange(len(tv_subjects)),
        test_size=val_fraction,
        stratify=tv_y,
        random_state=seed,
    )
    train_s0 = set(tv_subjects[tr_rel])
    val_s0 = set(tv_subjects[va_rel])
    tr0 = meta_df[meta_df["subject_id"].isin(train_s0)].copy()
    va0 = meta_df[meta_df["subject_id"].isin(val_s0)].copy()
    _, _, _, _, in_ch = build_loaders_from_meta(tr0, va0, None, pad_mode="zero", padding="fi

## Run K-fold evaluation


In [2]:
import os
import sys

root = os.path.abspath(os.getcwd())
if root not in sys.path:
    sys.path.insert(0, root)

from dysxai_cv_evaluation import run_kfold

# Example (adjust flags as needed):
run_kfold(n_folds=5, quick=True, scenarios="no_time", output_csv="kfold_leakage_results.csv")


  [time_zero] fold 2/5  Val AUC 1.0000  Test AUC 1.0000


  [time_zero] fold 3/5  Val AUC 0.9091  Test AUC 1.0000


  [time_zero] fold 4/5  Val AUC 1.0000  Test AUC 0.8531


  [time_zero] fold 5/5  Val AUC 1.0000  Test AUC 0.7832
  [time_zero] MEAN Test AUC: 0.9273  std 0.0918  (n_folds=5)


  [time_rep] fold 1/5  Val AUC 0.9394  Test AUC 0.9231


...

  [no_time] fold 1/5  Val AUC 0.6667  Test AUC 0.8910


  [no_time] fold 2/5  Val AUC 0.8990  Test AUC 0.9792


  [no_time] fold 3/5  Val AUC 0.6545  Test AUC 0.9720


  [no_time] fold 4/5  Val AUC 0.9091  Test AUC 0.8042


  [no_time] fold 5/5  Val AUC 0.9091  Test AUC 0.8811
  [no_time] MEAN Test AUC: 0.9055  std 0.0647  (n_folds=5)
Wrote c:\Users\tiama\OneDrive\Desktop\Reserach Project\DysXAI\kfold_leakage_results.csv
